# STEP 3 · 변화 비교 (2023 to 2025)

목적: 시계열로 **"AI는 늘었는데 부담도 늘었다"** 를 보인다.

**평균보다 비율(%)로 말하기.**

In [1]:
import sys; sys.path.insert(0,'..')
import pyreadstat, pandas as pd, numpy as np
import warnings; warnings.filterwarnings('ignore')

df23,_ = pyreadstat.read_sav("../data/raw/journalist_2023.sav")
df25,_ = pyreadstat.read_sav("../data/raw/journalist_2025.sav")

def build(df, year):
    """원본을 받아 분석용 파생변수를 붙여 돌려준다."""
    d = df.copy()
    if year==2025:
        d['fatigue']=d['q5']
        d['ai_user']=(d['q38']==1)
        d['benefit']=d[['q39_1','q39_2','q39_3']].mean(1)
        d['risk']=d[['q39_4','q39_5','q39_6']].mean(1)
        d['burnout']=d[['q21_3','q21_4']].mean(1)
        d['satis']=d['q19_1']
    else:
        d['fatigue']=d['q6']
        d['ai_user']=d['q43_9'].isna()
        d['benefit']=d[['q45_1','q45_2','q45_3']].mean(1)
        d['risk']=d[['q45_4','q45_5']].mean(1)
        d['burnout']=d[['q26_3','q26_4']].mean(1)
        d['satis']=d['q24_1']
    d['role_imp']=d[[f'q2_{i}' for i in range(1,8)]].mean(1)
    d['role_perf']=d[[f'q3_{i}' for i in range(1,8)]].mean(1)
    d['role_gap']=d['role_imp']-d['role_perf']
    return d

d23 = build(df23,2023)
d25 = build(df25,2025)

### 핵심 5지표 변화

In [2]:
def pct_high(s): return (s>=4).mean()*100

rows = [
  ['AI 활용률(%)',      d23['ai_user'].mean()*100,  d25['ai_user'].mean()*100],
  ['디지털 피로 4-5(%)', pct_high(d23['fatigue']),    pct_high(d25['fatigue'])],
  ['직업 만족(10점평균)', d23['satis'].mean(),         d25['satis'].mean()],
  ['탈진 호소 4-5(%)',  pct_high(d23['burnout']),    pct_high(d25['burnout'])],
  ['역할갭(중요-실행)',  d23['role_gap'].mean(),      d25['role_gap'].mean()],
]
chg = pd.DataFrame(rows, columns=['지표','2023','2025'])
chg['변화'] = (chg['2025']-chg['2023']).round(2)
chg = chg.round(2)
chg.to_csv("../outputs/tables/change_2023_2025.csv", index=False, encoding='utf-8-sig')
chg

,지표,2023,2025,변화
0,AI 활용률(%),54.30,58.07,3.77
1,디지털 피로 4-5(%),38.29,45.50,7.21
2,직업 만족(10점평균),6.09,6.06,-0.03
3,탈진 호소 4-5(%),30.08,31.58,1.50
4,역할갭(중요-실행),0.81,1.08,0.27


### 결과 해석 - 첫 번째 펀치라인

- **AI 활용률 +3.8%p** (54.3 -> 58.1)
- **디지털 피로 +7.2%p** (38.3 -> 45.5)  <- 활용 증가의 **약 2배 속도**
- 직업 만족도는 6.09 -> 6.06 **정체**
- 역할갭은 0.81 -> 1.08 **확대**

> "AI는 빠르게 자리 잡았지만, 그것이 직업적 안정감으로 이어지지
> 않았다. 도구는 늘었으나 피로가 더 빨리 쌓였다."
